## Phase 5 — Embeddings + Vector Search Index
**Reads from:** `main.silver.news_articles`
**Creates:**
- `main.silver.news_for_search` — CDF-enabled source table for Vector Search
- Vector Search endpoint: `stock-assistant-vs`
- Vector Search index: `main.silver.news_for_search_index`

Uses Databricks Managed Embeddings (no manual embedding calls needed).
The index auto-syncs when new rows arrive in the source Delta table.


In [ ]:
# 0. Imports and config
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from databricks.vector_search.client import VectorSearchClient
from datetime import datetime
import time

spark = SparkSession.builder.getOrCreate()

PROCESSED_AT    = datetime.now().isoformat()
ENDPOINT_NAME   = "stock-assistant-vs"
INDEX_NAME      = "main.silver.news_for_search_index"
SOURCE_TABLE    = "main.silver.news_for_search"
EMBEDDING_MODEL = "databricks-bge-large-en"   # Free Edition supported model

print(f"Started: {PROCESSED_AT}")
print(f"Endpoint : {ENDPOINT_NAME}")
print(f"Index    : {INDEX_NAME}")
print(f"Model    : {EMBEDDING_MODEL}")


In [ ]:
# 1. Prepare source table for Vector Search
# Vector Search requires:
#   - A primary key column (article_id)
#   - Change Data Feed enabled (delta.enableChangeDataFeed = true)
#   - A text column to embed (search_text)
print("\n--- Preparing source table for Vector Search ---")

silver_news = spark.table("main.silver.news_articles")

# Build a rich search_text field combining title + description
# This gives the embedding model more context per article
news_for_search = (
    silver_news
    .withColumn("search_text",
        F.concat_ws(" | ",
            F.col("ticker"),
            F.col("title"),
            F.coalesce(F.col("description"), F.lit(""))
        )
    )
    .select(
        "article_id",       # Primary key — required by Vector Search
        "ticker",
        "title",
        "description",
        "search_text",      # Column to embed
        "publisher_name",
        "sentiment",
        "published_utc",
        "published_ts",
        "article_age_days",
        "has_sentiment",
        "processed_at"
    )
    .filter(F.col("article_id").isNotNull())
    .filter(F.col("search_text").isNotNull())
)

# Write with Change Data Feed enabled (required for Delta Sync index)
(news_for_search
 .write.format("delta")
 .mode("overwrite")
 .option("overwriteSchema", "true")
 .saveAsTable(SOURCE_TABLE))

# Enable CDF on the table (required for auto-sync Vector Search index)
spark.sql(f"""
    ALTER TABLE {SOURCE_TABLE}
    SET TBLPROPERTIES (delta.enableChangeDataFeed = true)
""")

count = spark.table(SOURCE_TABLE).count()
print(f"Written {count} rows → {SOURCE_TABLE} ✓")
print("CDF enabled ✓")

print("\nSample search_text values:")
spark.table(SOURCE_TABLE) \
     .select("article_id", "ticker", "search_text") \
     .show(3, truncate=80)


In [ ]:
# 2. Create Vector Search Endpoint
# Free Edition allows 1 endpoint — check if it already exists first
print("\n--- Setting up Vector Search endpoint ---")

vsc = VectorSearchClient()

# Check existing endpoints
try:
    existing = vsc.get_endpoint(ENDPOINT_NAME)
    status   = existing.get("endpoint_status", {}).get("state", "UNKNOWN")
    print(f"Endpoint '{ENDPOINT_NAME}' already exists — status: {status}")
    if status != "ONLINE":
        print("Waiting for endpoint to come online...")
        for _ in range(20):
            time.sleep(30)
            status = vsc.get_endpoint(ENDPOINT_NAME) \
                        .get("endpoint_status", {}).get("state", "UNKNOWN")
            print(f"  Status: {status}")
            if status == "ONLINE":
                break
except Exception:
    print(f"Creating endpoint '{ENDPOINT_NAME}'...")
    vsc.create_endpoint(
        name          = ENDPOINT_NAME,
        endpoint_type = "STANDARD"
    )
    print("Endpoint creation started — waiting for ONLINE status (~3-5 min)...")
    for _ in range(20):
        time.sleep(30)
        try:
            status = vsc.get_endpoint(ENDPOINT_NAME) \
                        .get("endpoint_status", {}).get("state", "UNKNOWN")
            print(f"  Status: {status}")
            if status == "ONLINE":
                break
        except Exception as e:
            print(f"  Waiting... ({e})")

print(f"\nEndpoint '{ENDPOINT_NAME}' is ready ✓")


In [ ]:
# 3. Create Vector Search Index (Delta Sync with Managed Embeddings)
# Delta Sync = index auto-updates when source table changes via CDF
# Managed Embeddings = Databricks generates embeddings automatically
print("\n--- Creating Vector Search index ---")

try:
    existing_index = vsc.get_index(ENDPOINT_NAME, INDEX_NAME)
    print(f"Index '{INDEX_NAME}' already exists")
    print(f"Status: {existing_index.get('status', {}).get('ready', False)}")
except Exception:
    print(f"Creating index '{INDEX_NAME}'...")
    vsc.create_delta_sync_index(
        endpoint_name          = ENDPOINT_NAME,
        index_name             = INDEX_NAME,
        source_table_name      = SOURCE_TABLE,
        pipeline_type          = "TRIGGERED",     # Manual sync trigger
        primary_key            = "article_id",
        embedding_source_column= "search_text",   # Column to embed
        embedding_model_endpoint_name = EMBEDDING_MODEL
    )
    print("Index creation started — waiting for ONLINE status (~3-5 min)...")
    for _ in range(20):
        time.sleep(30)
        try:
            idx    = vsc.get_index(ENDPOINT_NAME, INDEX_NAME)
            ready  = idx.get("status", {}).get("ready", False)
            detail = idx.get("status", {}).get("message", "")
            print(f"  Ready: {ready}  {detail}")
            if ready:
                break
        except Exception as e:
            print(f"  Waiting... ({e})")

print(f"\nIndex '{INDEX_NAME}' is ready ✓")


In [ ]:
# 4. Sync the index (trigger first pipeline run)
print("\n--- Triggering index sync ---")

try:
    idx = vsc.get_index(ENDPOINT_NAME, INDEX_NAME)
    idx.sync()
    print("Index sync triggered ✓")
    print("Waiting 60s for sync to complete...")
    time.sleep(60)
except Exception as e:
    print(f"Sync note: {e}")


In [ ]:
# 5. Test semantic search
print("\n--- Testing semantic search ---")

test_queries = [
    "Apple stock price drop earnings",
    "AI artificial intelligence technology growth",
    "Federal Reserve interest rates inflation",
]

try:
    idx = vsc.get_index(ENDPOINT_NAME, INDEX_NAME)

    for query in test_queries:
        print(f"\nQuery: '{query}'")
        results = idx.similarity_search(
            query_text        = query,
            columns           = ["article_id", "ticker", "title",
                                  "sentiment", "publisher_name"],
            num_results       = 3
        )
        hits = results.get("result", {}).get("data_array", [])
        for i, hit in enumerate(hits, 1):
            print(f"  {i}. [{hit[1]}] {hit[2][:70]}... ({hit[3]})")

except Exception as e:
    print(f"Search test note: {e}")
    print("Index may still be syncing — re-run this cell in 2 minutes")


In [ ]:
# 6. Summary
print("\n=== Embeddings + Vector Search Summary ===")
print(f"Processed at  : {PROCESSED_AT}")
print(f"Source table  : {SOURCE_TABLE}")
print(f"VS Endpoint   : {ENDPOINT_NAME}")
print(f"VS Index      : {INDEX_NAME}")
print(f"Embed model   : {EMBEDDING_MODEL}")
print(f"Rows indexed  : {spark.table(SOURCE_TABLE).count()}")
print("""
How it works:
  1. news_for_search Delta table (CDF enabled)
       ↓ auto-sync via Change Data Feed
  2. Vector Search index (managed embeddings)
       ↓ semantic similarity search
  3. AI Agent tool: search_news(query)
       ↓ returns top-k relevant articles
  4. Agent uses articles as RAG context
       ↓ grounds responses in real news
""")
print("Phase 5 complete ✓")
